# Stochastic Block Model
## Star Wars II: Attack of the Clones

Son los personajes de Star Wars: Episode II - Attack of the Clones, conectados si comparten escenas; las aristas tienen pesos igual a la cantidad de escenas que comparten.

In [1]:
import html
import math
import random
import unicodedata
import xml.etree.ElementTree as ET
from pathlib import Path

import networkx as nx
from IPython.display import HTML, display


In [2]:
def find_repo_file(relative_path):
    relative_path = Path(relative_path)
    candidates = [Path.cwd(), *Path.cwd().parents]
    for base in candidates:
        candidate = base / relative_path
        if candidate.exists():
            return candidate
    raise FileNotFoundError(f"No se encontro {relative_path} subiendo desde {Path.cwd()}")


def ascii_text(value):
    value = unicodedata.normalize("NFD", str(value))
    return "".join(ch for ch in value if unicodedata.category(ch) != "Mn")


def load_gexf_simple_undirected(path):
    """Carga nodos/aristas de un GEXF de Gephi y conserva las etiquetas legibles."""
    namespace = {"g": "http://www.gexf.net/1.2draft"}
    root = ET.parse(path).getroot()

    labels = {
        node.attrib["id"]: ascii_text(node.attrib.get("label", node.attrib["id"]))
        for node in root.findall(".//g:node", namespace)
    }

    G = nx.Graph()
    G.add_nodes_from(labels.values())
    for edge in root.findall(".//g:edge", namespace):
        u = labels[edge.attrib["source"]]
        v = labels[edge.attrib["target"]]
        if u != v:
            G.add_edge(u, v)
    return G

movie_id = 774
gexf_path = find_repo_file(Path("data/dataverse/gexf") / f"{movie_id}.gexf")
G1 = load_gexf_simple_undirected(gexf_path)
print("loaded", movie_id, "| Nodos:", G1.number_of_nodes(), "| Aristas:", G1.number_of_edges())


loaded 774 | Nodos: 47 | Aristas: 148


In [3]:
def show_table(rows, columns=None, float_digits=4):
    if not rows:
        display(HTML("<em>Sin datos.</em>"))
        return

    if columns is None:
        columns = list(rows[0].keys())

    def fmt(value):
        if isinstance(value, float):
            return f"{value:.{float_digits}f}"
        return html.escape(str(value))

    header = "".join(f"<th>{html.escape(str(col))}</th>" for col in columns)
    body = "".join(
        "<tr>" + "".join(f"<td>{fmt(row.get(col, ''))}</td>" for col in columns) + "</tr>"
        for row in rows
    )
    display(HTML(f"""
    <table style="border-collapse:collapse; font-size:14px; max-width:100%;">
      <thead><tr style="background:#f2f2f2;">{header}</tr></thead>
      <tbody>{body}</tbody>
    </table>
    <style>
      table td, table th {{ border:1px solid #ddd; padding:5px 8px; vertical-align:top; }}
      table th {{ text-align:left; }}
    </style>
    """))


def graph_summary(G):
    components = sorted(nx.connected_components(G), key=len, reverse=True)
    return [{
        "nodos": G.number_of_nodes(),
        "aristas": G.number_of_edges(),
        "densidad": nx.density(G),
        "componentes": len(components),
        "tamano_componente_mayor": len(components[0]) if components else 0,
        "pares_sin_arista": len(list(nx.non_edges(G)))
    }]


def top_degrees(G, k=10):
    return [
        {"vertice": node, "grado": degree}
        for node, degree in sorted(G.degree, key=lambda item: item[1], reverse=True)[:k]
    ]


def modularity_partition(G):
    communities = list(nx.algorithms.community.greedy_modularity_communities(G))
    communities = [sorted(list(c), key=str) for c in communities]
    communities.sort(key=lambda c: (-len(c), str(c[0]) if c else ""))
    community_of = {}
    for idx, community in enumerate(communities):
        for node in community:
            community_of[node] = idx
    return communities, community_of


def _scale_positions(pos, width, height, margin):
    return _scale_positions_box(pos, width, height, margin, margin, margin, margin)


def _scale_positions_box(pos, width, height, left, right, top, bottom):
    xs = [xy[0] for xy in pos.values()]
    ys = [xy[1] for xy in pos.values()]
    min_x, max_x = min(xs), max(xs)
    min_y, max_y = min(ys), max(ys)
    span_x = max(max_x - min_x, 1e-9)
    span_y = max(max_y - min_y, 1e-9)
    return {
        node: (
            left + (xy[0] - min_x) / span_x * (width - left - right),
            top + (xy[1] - min_y) / span_y * (height - top - bottom),
        )
        for node, xy in pos.items()
    }


def modularity_seed_layout(G):
    if "blocks" in globals() and "block_of" in globals():
        communities, community_of = blocks, block_of
    else:
        communities, community_of, _ = infer_sbm_blocks(G)
    pos = {}
    n_com = max(len(communities), 1)
    for ci, community in enumerate(communities):
        center_angle = 2 * math.pi * ci / n_com
        cx = 2.8 * math.cos(center_angle)
        cy = 2.8 * math.sin(center_angle)
        inner_n = max(len(community), 1)
        inner_radius = 0.42 + 0.055 * inner_n
        for j, node in enumerate(community):
            angle = 2 * math.pi * j / inner_n + 0.37 * ci
            pos[node] = (cx + inner_radius * math.cos(angle), cy + inner_radius * math.sin(angle))
    return pos, communities, community_of


def force_layout(G, iterations=180):
    pos, communities, community_of = modularity_seed_layout(G)
    nodes = list(G.nodes())
    n = max(len(nodes), 1)
    area = 32.0
    k = math.sqrt(area / n)
    edges = list(G.edges())

    for step in range(iterations):
        disp = {node: [0.0, 0.0] for node in nodes}
        temp = 0.22 * (1 - step / iterations) + 0.018

        for i, u in enumerate(nodes):
            ux, uy = pos[u]
            for v in nodes[i + 1:]:
                vx, vy = pos[v]
                dx = ux - vx
                dy = uy - vy
                dist = math.sqrt(dx * dx + dy * dy) + 1e-6
                force = (k * k) / dist
                fx = dx / dist * force
                fy = dy / dist * force
                disp[u][0] += fx
                disp[u][1] += fy
                disp[v][0] -= fx
                disp[v][1] -= fy

        for u, v in edges:
            ux, uy = pos[u]
            vx, vy = pos[v]
            dx = ux - vx
            dy = uy - vy
            dist = math.sqrt(dx * dx + dy * dy) + 1e-6
            same = community_of.get(u) == community_of.get(v)
            strength = 1.55 if same else 0.75
            force = strength * (dist * dist) / k
            fx = dx / dist * force
            fy = dy / dist * force
            disp[u][0] -= fx
            disp[u][1] -= fy
            disp[v][0] += fx
            disp[v][1] += fy

        for node in nodes:
            dx, dy = disp[node]
            length = math.sqrt(dx * dx + dy * dy) + 1e-6
            x, y = pos[node]
            pos[node] = (x + dx / length * min(length, temp), y + dy / length * min(length, temp))
    return pos, communities, community_of


def show_network_svg(G, title="Red", predicted=None, width=980, height=780, label_mode="auto"):
    predicted = predicted or []
    raw_pos, communities, community_of = force_layout(G)
    pos = _scale_positions_box(raw_pos, width, height, left=78, right=54, top=170, bottom=52)
    degrees = dict(G.degree())
    max_degree = max(degrees.values()) if degrees else 1
    palette = ["#4e79a7", "#f28e2b", "#59a14f", "#e15759", "#76b7b2", "#edc948", "#b07aa1", "#ff9da7", "#9c755f", "#bab0ab"]

    predicted_nodes = {node for row in predicted for node in (row["u"], row["v"])}
    top_label_nodes = {node for node, _ in sorted(G.degree, key=lambda item: item[1], reverse=True)[:10]}
    subtitle = f"{G.number_of_nodes()} nodos | {G.number_of_edges()} aristas observadas | {len(communities)} bloques estimados"

    community_parts = []
    for ci, community in enumerate(communities):
        xs = [pos[node][0] for node in community]
        ys = [pos[node][1] for node in community]
        if not xs:
            continue
        cx = sum(xs) / len(xs)
        cy = sum(ys) / len(ys)
        radius = max([math.sqrt((x - cx) ** 2 + (y - cy) ** 2) for x, y in zip(xs, ys)] + [22]) + 40
        color = palette[ci % len(palette)]
        community_parts.append(
            f'<circle cx="{cx:.1f}" cy="{cy:.1f}" r="{radius:.1f}" fill="{color}" opacity="0.075" stroke="{color}" stroke-width="1.3" stroke-dasharray="5 5" />'
        )
        community_parts.append(
            f'<text x="{cx:.1f}" y="{max(cy - radius + 18, 158):.1f}" font-size="12" text-anchor="middle" font-family="Arial" fill="{color}">B{ci} ({len(community)} nodos)</text>'
        )

    edge_parts = []
    for u, v in G.edges():
        x1, y1 = pos[u]
        x2, y2 = pos[v]
        same = community_of.get(u) == community_of.get(v)
        stroke = "#a9b1bb" if same else "#7d8793"
        opacity = "0.42" if same else "0.68"
        width_line = "1.0" if same else "1.6"
        edge_parts.append(
            f'<line x1="{x1:.1f}" y1="{y1:.1f}" x2="{x2:.1f}" y2="{y2:.1f}" stroke="{stroke}" stroke-width="{width_line}" opacity="{opacity}" />'
        )

    pred_parts = []
    if predicted:
        scores = [row["puntaje"] for row in predicted]
        min_s, max_s = min(scores), max(scores)
    else:
        min_s, max_s = 0, 1
    for row in predicted:
        u, v = row["u"], row["v"]
        if u not in pos or v not in pos:
            continue
        x1, y1 = pos[u]
        x2, y2 = pos[v]
        score = row["puntaje"]
        same = community_of.get(u) == community_of.get(v)
        color = palette[community_of.get(u, 0) % len(palette)] if same else "#d62728"
        width_line = 2.4 + 3.2 * ((score - min_s) / (max_s - min_s + 1e-9))
        pred_parts.append(
            f'<line x1="{x1:.1f}" y1="{y1:.1f}" x2="{x2:.1f}" y2="{y2:.1f}" stroke="{color}" stroke-width="{width_line:.1f}" opacity="0.97">'
            f'<title>{html.escape(str(u))} - {html.escape(str(v))}: {score:.4f} | {"intra-bloque" if same else "inter-bloque"}</title></line>'
        )

    node_parts = []
    label_parts = []
    for node in G.nodes():
        x, y = pos[node]
        r = 7 + 14 * degrees[node] / max_degree
        color = palette[community_of.get(node, 0) % len(palette)]
        node_parts.append(
            f'<circle cx="{x:.1f}" cy="{y:.1f}" r="{r:.1f}" fill="{color}" stroke="#222" stroke-width="1.1">'
            f'<title>{html.escape(str(node))} | grado {degrees[node]} | bloque B{community_of.get(node, -1)}</title></circle>'
        )
        should_label = label_mode == "all" or node in predicted_nodes or node in top_label_nodes
        if should_label:
            label = html.escape(str(node))
            label_parts.append(
                f'<text x="{x:.1f}" y="{y-r-5:.1f}" font-size="10" text-anchor="middle" font-family="Arial, sans-serif" fill="#1b1b1b">{label}</text>'
            )

    community_legend = []
    for ci, community in enumerate(communities[:6]):
        color = palette[ci % len(palette)]
        x = 565 + (ci % 3) * 122
        y = 88 + (ci // 3) * 26
        community_legend.append(
            f'<circle cx="{x:.1f}" cy="{y:.1f}" r="6" fill="{color}" stroke="#222" stroke-width="0.6" />'
            f'<text x="{x+12:.1f}" y="{y+4:.1f}" font-size="12" font-family="Arial" fill="#333">B{ci}: {len(community)}</text>'
        )

    svg = f"""
    <div style="max-width:{width}px; overflow-x:auto; font-family:Arial, sans-serif;">
      <svg viewBox="0 0 {width} {height}" width="100%" height="auto" role="img" aria-label="{html.escape(title)}">
        <title>{html.escape(title)}</title>
        <desc>{html.escape(subtitle)}. Los colores indican bloques; las lineas gruesas son pares SBM resaltados.</desc>
        <rect width="100%" height="100%" fill="#ffffff" />

        <text x="24" y="32" font-size="22" font-family="Arial, sans-serif" font-weight="700" fill="#111">{html.escape(title)}</text>
        <text x="24" y="56" font-size="13" font-family="Arial, sans-serif" fill="#555">{html.escape(subtitle)}</text>

        <rect x="20" y="70" width="930" height="54" rx="8" fill="#f8f9fb" stroke="#d6dbe1" stroke-width="1" />
        <text x="34" y="92" font-size="13" font-family="Arial" font-weight="700" fill="#222">Leyenda</text>
        <line x1="34" y1="110" x2="86" y2="110" stroke="#a9b1bb" stroke-width="1.6" opacity="0.8" />
        <text x="96" y="114" font-size="12" font-family="Arial" fill="#333">Coaparición</text>
        <line x1="220" y1="110" x2="272" y2="110" stroke="#4e79a7" stroke-width="4" />
        <text x="282" y="114" font-size="12" font-family="Arial" fill="#333">Par SBM dentro del bloque</text>
        <line x1="545" y1="110" x2="597" y2="110" stroke="#d62728" stroke-width="4" />
        <text x="607" y="114" font-size="12" font-family="Arial" fill="#333">Par SBM entre bloques</text>
        <circle cx="780" cy="110" r="5" fill="#4e79a7" stroke="#222" stroke-width="0.8" />
        <circle cx="798" cy="110" r="10" fill="#4e79a7" stroke="#222" stroke-width="0.8" />
        <text x="814" y="114" font-size="12" font-family="Arial" fill="#333">Tamaño del nodo = grado</text>

        <g>{''.join(community_parts)}</g>
        <g>{''.join(edge_parts)}</g>
        <g>{''.join(pred_parts)}</g>
        <g>{''.join(node_parts)}</g>
        <g>{''.join(label_parts)}</g>
      </svg>
    </div>
    """
    display(HTML(svg))


def _partition_from_labels(nodes, labels):
    grouped = {}
    for node, label in zip(nodes, labels):
        grouped.setdefault(int(label), []).append(node)
    blocks = [sorted(group, key=str) for group in grouped.values()]
    blocks.sort(key=lambda c: (-len(c), str(c[0]) if c else ""))
    block_of = {node: idx for idx, block in enumerate(blocks) for node in block}
    return blocks, block_of


def _adjacency_matrix(G, nodes):
    idx = {node: i for i, node in enumerate(nodes)}
    A = [[0.0 for _ in nodes] for _ in nodes]
    for u, v in G.edges():
        i, j = idx[u], idx[v]
        A[i][j] = A[j][i] = 1.0
    return A


def _jacobi_eigen_symmetric(A, max_iter=7000, tol=1e-10):
    n = len(A)
    M = [row[:] for row in A]
    V = [[1.0 if i == j else 0.0 for j in range(n)] for i in range(n)]
    for _ in range(max_iter):
        p, q, max_off = 0, 1, 0.0
        for i in range(n):
            for j in range(i + 1, n):
                if abs(M[i][j]) > max_off:
                    p, q, max_off = i, j, abs(M[i][j])
        if max_off < tol:
            break
        angle = math.pi / 4 if abs(M[p][p] - M[q][q]) < 1e-15 else 0.5 * math.atan2(2 * M[p][q], M[q][q] - M[p][p])
        c, s = math.cos(angle), math.sin(angle)
        for i in range(n):
            if i != p and i != q:
                mip, miq = M[i][p], M[i][q]
                M[i][p] = M[p][i] = c * mip - s * miq
                M[i][q] = M[q][i] = s * mip + c * miq
        mpp, mqq, mpq = M[p][p], M[q][q], M[p][q]
        M[p][p] = c * c * mpp - 2 * s * c * mpq + s * s * mqq
        M[q][q] = s * s * mpp + 2 * s * c * mpq + c * c * mqq
        M[p][q] = M[q][p] = 0.0
        for i in range(n):
            vip, viq = V[i][p], V[i][q]
            V[i][p] = c * vip - s * viq
            V[i][q] = s * vip + c * viq
    return [M[i][i] for i in range(n)], V


def _squared_distance(x, y):
    return sum((a - b) ** 2 for a, b in zip(x, y))


def _kmeans(points, k, seed=1121, n_init=45, max_iter=100):
    rng = random.Random(seed)
    n = len(points)
    best_labels, best_inertia = None, float("inf")
    for _ in range(n_init):
        centers = [points[i][:] for i in rng.sample(range(n), k)]
        labels = [-1] * n
        for _ in range(max_iter):
            new_labels = [min(range(k), key=lambda c: _squared_distance(point, centers[c])) for point in points]
            counts = [new_labels.count(c) for c in range(k)]
            for empty in [c for c in range(k) if counts[c] == 0]:
                donor = max(range(k), key=lambda c: counts[c])
                donor_points = [i for i, label in enumerate(new_labels) if label == donor]
                farthest = max(donor_points, key=lambda i: _squared_distance(points[i], centers[donor]))
                new_labels[farthest] = empty
                counts[donor] -= 1
                counts[empty] += 1
            new_centers = []
            for c in range(k):
                members = [points[i] for i, label in enumerate(new_labels) if label == c]
                new_centers.append([sum(values) / len(members) for values in zip(*members)])
            if new_labels == labels:
                break
            labels, centers = new_labels, new_centers
        inertia = sum(_squared_distance(points[i], centers[labels[i]]) for i in range(n))
        if inertia < best_inertia:
            best_labels, best_inertia = labels[:], inertia
    return best_labels


def _spectral_block_labels(G, k):
    nodes = sorted(G.nodes(), key=str)
    if k == 1:
        return nodes, [0] * len(nodes)
    A = _adjacency_matrix(G, nodes)
    values, vectors = _jacobi_eigen_symmetric(A)
    order = sorted(range(len(nodes)), key=lambda i: abs(values[i]), reverse=True)[:k]
    points = []
    for row in range(len(nodes)):
        point = [vectors[row][col] * math.sqrt(abs(values[col]) + 1e-12) for col in order]
        norm = math.sqrt(sum(x * x for x in point))
        points.append([x / norm for x in point] if norm > 1e-12 else point)
    return nodes, _kmeans(points, k)


def _bernoulli_sbm_log_likelihood(G, blocks):
    eps = 1e-9
    ll = 0.0
    for r, block_r in enumerate(blocks):
        for s, block_s in enumerate(blocks[r:], start=r):
            if r == s:
                possible = len(block_r) * (len(block_r) - 1) // 2
                observed = G.subgraph(block_r).number_of_edges()
            else:
                possible = len(block_r) * len(block_s)
                observed = sum(1 for u in block_r for v in block_s if G.has_edge(u, v))
            if possible:
                p = min(max(observed / possible, eps), 1 - eps)
                ll += observed * math.log(p) + (possible - observed) * math.log(1 - p)
    return ll


def _bic_for_partition(G, blocks):
    k = len(blocks)
    n_pairs = max(G.number_of_nodes() * (G.number_of_nodes() - 1) // 2, 1)
    ll = _bernoulli_sbm_log_likelihood(G, blocks)
    parameters = k * (k + 1) / 2 + (k - 1)
    return ll, -2 * ll + parameters * math.log(n_pairs)


def _infer_sbm_blocks_spectral_bic(G, k_min=2, k_max=8):
    candidates = []
    for k in range(k_min, min(k_max, G.number_of_nodes() - 1) + 1):
        nodes, labels = _spectral_block_labels(G, k)
        candidate_blocks, candidate_block_of = _partition_from_labels(nodes, labels)
        ll, bic = _bic_for_partition(G, candidate_blocks)
        candidates.append({"K": len(candidate_blocks), "blocks": candidate_blocks, "block_of": candidate_block_of, "log_likelihood": ll, "BIC": bic})
    best = min(candidates, key=lambda row: row["BIC"])
    return best["blocks"], best["block_of"], {
        "method": "fallback_espectral_bic",
        "method_label": "Fallback espectral + BIC para SBM Bernoulli",
        "is_exact_sbm": False,
        "k_selection": "inferido automaticamente por BIC",
        "k_candidates": [{"K": row["K"], "log_likelihood": row["log_likelihood"], "BIC": row["BIC"]} for row in candidates],
    }


def _infer_sbm_blocks_graph_tool(G):
    import graph_tool.all as gt
    from graph_tool.inference import minimize_blockmodel_dl
    nodes = sorted(G.nodes(), key=str)
    idx = {node: i for i, node in enumerate(nodes)}
    g = gt.Graph(directed=False)
    g.add_vertex(len(nodes))
    for u, v in G.edges():
        g.add_edge(g.vertex(idx[u]), g.vertex(idx[v]))
    state = minimize_blockmodel_dl(g)
    labels = [int(state.get_blocks()[g.vertex(i)]) for i in range(len(nodes))]
    blocks, block_of = _partition_from_labels(nodes, labels)
    return blocks, block_of, {
        "method": "graph_tool_minimize_blockmodel_dl",
        "method_label": "graph-tool minimize_blockmodel_dl",
        "is_exact_sbm": True,
        "k_selection": "inferido automaticamente por minimizacion de description length",
        "description_length": float(state.entropy()),
    }


def infer_sbm_blocks(G):
    try:
        return _infer_sbm_blocks_graph_tool(G)
    except Exception as exc:
        inferred_blocks, inferred_block_of, metadata = _infer_sbm_blocks_spectral_bic(G)
        metadata["graph_tool_status"] = f"graph-tool no disponible o fallo: {type(exc).__name__}: {exc}"
        return inferred_blocks, inferred_block_of, metadata


def sbm_blocks(G):
    return infer_sbm_blocks(G)


def sbm_probability_matrix(G, blocks):
    matrix = []
    for r, block_r in enumerate(blocks):
        row = []
        for s, block_s in enumerate(blocks):
            if r == s:
                possible = len(block_r) * (len(block_r) - 1) // 2
                observed = G.subgraph(block_r).number_of_edges()
            else:
                possible = len(block_r) * len(block_s)
                observed = sum(1 for u in block_r for v in block_s if G.has_edge(u, v))
            probability = observed / possible if possible else 0.0
            row.append({"bloque_i": r, "bloque_j": s, "aristas": observed, "posibles": possible, "p": probability})
        matrix.append(row)
    return matrix


def block_summary_table(G, blocks):
    rows = []
    total_edges = G.number_of_edges()
    for i, block in enumerate(blocks):
        subgraph = G.subgraph(block)
        internal_edges = subgraph.number_of_edges()
        possible = len(block) * (len(block) - 1) // 2
        rows.append({
            "bloque": f"B{i}",
            "nodos": len(block),
            "aristas_internas": internal_edges,
            "posibles_internas": possible,
            "p_interna": internal_edges / possible if possible else 0.0,
            "porcentaje_aristas": internal_edges / total_edges if total_edges else 0.0,
            "personajes": ", ".join(block),
        })
    return rows


def block_probability_table(G, blocks):
    rows = []
    for row in sbm_probability_matrix(G, blocks):
        for cell in row:
            rows.append({
                "bloque_i": f"B{cell['bloque_i']}",
                "bloque_j": f"B{cell['bloque_j']}",
                "aristas_observadas": cell["aristas"],
                "aristas_posibles": cell["posibles"],
                "probabilidad_sbm": cell["p"],
            })
    return rows


def sbm_missing_edges(G, blocks, block_of, k=12):
    matrix = sbm_probability_matrix(G, blocks)
    rows = []
    for u, v in nx.non_edges(G):
        bu = block_of[u]
        bv = block_of[v]
        probability = matrix[bu][bv]["p"]
        rows.append({
            "u": u,
            "v": v,
            "puntaje": probability,
            "bloque_u": f"B{bu}",
            "bloque_v": f"B{bv}",
            "tipo": "intra-bloque" if bu == bv else "inter-bloque",
        })
    rows.sort(key=lambda row: (row["puntaje"], row["tipo"] == "intra-bloque", row["u"], row["v"]), reverse=True)
    return rows[:k]


def observed_block_bridges(G, block_of, k=12):
    rows = []
    for u, v in G.edges():
        bu = block_of[u]
        bv = block_of[v]
        if bu != bv:
            rows.append({
                "u": u,
                "v": v,
                "bloque_u": f"B{bu}",
                "bloque_v": f"B{bv}",
                "grado_u+grado_v": G.degree[u] + G.degree[v],
            })
    rows.sort(key=lambda row: row["grado_u+grado_v"], reverse=True)
    return rows[:k]



def adjusted_rand_index(labels_a, labels_b):
    nodes = sorted(labels_a)
    n = len(nodes)
    choose2 = lambda x: x * (x - 1) / 2
    table, counts_a, counts_b = {}, {}, {}
    for node in nodes:
        a, b = labels_a[node], labels_b[node]
        table[(a, b)] = table.get((a, b), 0) + 1
        counts_a[a] = counts_a.get(a, 0) + 1
        counts_b[b] = counts_b.get(b, 0) + 1
    sum_comb = sum(choose2(v) for v in table.values())
    sum_a = sum(choose2(v) for v in counts_a.values())
    sum_b = sum(choose2(v) for v in counts_b.values())
    total = choose2(n)
    expected = sum_a * sum_b / total if total else 0
    maximum = 0.5 * (sum_a + sum_b)
    return (sum_comb - expected) / (maximum - expected) if maximum != expected else 1.0


def normalized_mutual_info(labels_a, labels_b):
    nodes = sorted(labels_a)
    n = len(nodes)
    table, counts_a, counts_b = {}, {}, {}
    for node in nodes:
        a, b = labels_a[node], labels_b[node]
        table[(a, b)] = table.get((a, b), 0) + 1
        counts_a[a] = counts_a.get(a, 0) + 1
        counts_b[b] = counts_b.get(b, 0) + 1
    mi = sum((count / n) * math.log((count * n) / (counts_a[a] * counts_b[b])) for (a, b), count in table.items())
    ha = -sum((count / n) * math.log(count / n) for count in counts_a.values())
    hb = -sum((count / n) * math.log(count / n) for count in counts_b.values())
    return mi / math.sqrt(ha * hb) if ha > 0 and hb > 0 else 1.0


def comparison_table(G, sbm_blocks_, sbm_block_of, modularity_blocks, modularity_of, metadata):
    sbm_labels = {node: sbm_block_of[node] for node in G.nodes()}
    modularity_labels = {node: modularity_of[node] for node in G.nodes()}
    ari = adjusted_rand_index(sbm_labels, modularity_labels)
    nmi = normalized_mutual_info(sbm_labels, modularity_labels)
    return [{
        "particion": "SBM",
        "metodo": metadata["method_label"],
        "bloques": len(sbm_blocks_),
        "modularidad_Q": nx.algorithms.community.modularity(G, sbm_blocks_),
        "ARI_vs_modularidad": ari,
        "NMI_vs_modularidad": nmi,
    }, {
        "particion": "Modularidad",
        "metodo": "greedy_modularity_communities",
        "bloques": len(modularity_blocks),
        "modularidad_Q": nx.algorithms.community.modularity(G, modularity_blocks),
        "ARI_vs_modularidad": 1.0,
        "NMI_vs_modularidad": 1.0,
    }]

def show_block_heatmap_svg(G, blocks, width=620, cell=72):
    matrix = sbm_probability_matrix(G, blocks)
    n = len(blocks)
    left, top = 112, 88
    height = top + n * cell + 72
    max_p = max((cell_data["p"] for row in matrix for cell_data in row), default=1.0)
    parts = []
    for r, row in enumerate(matrix):
        for s, cell_data in enumerate(row):
            p = cell_data["p"]
            intensity = p / max_p if max_p else 0
            red = int(246 - 178 * intensity)
            green = int(248 - 119 * intensity)
            blue = int(250 - 65 * intensity)
            x = left + s * cell
            y = top + r * cell
            parts.append(
                f'<rect x="{x}" y="{y}" width="{cell}" height="{cell}" fill="rgb({red},{green},{blue})" stroke="#ffffff" stroke-width="2" />'
                f'<text x="{x + cell / 2}" y="{y + 32}" text-anchor="middle" font-size="15" font-family="Arial" font-weight="700" fill="#111">{p:.3f}</text>'
                f'<text x="{x + cell / 2}" y="{y + 51}" text-anchor="middle" font-size="10" font-family="Arial" fill="#333">{cell_data["aristas"]}/{cell_data["posibles"]}</text>'
            )
    labels = []
    for i, block in enumerate(blocks):
        x = left + i * cell + cell / 2
        y = top + i * cell + cell / 2
        labels.append(f'<text x="{x}" y="{top - 18}" text-anchor="middle" font-size="13" font-family="Arial" fill="#333">B{i}</text>')
        labels.append(f'<text x="{left - 18}" y="{y + 4}" text-anchor="end" font-size="13" font-family="Arial" fill="#333">B{i}</text>')
    svg = f'''
    <div style="max-width:{width}px; overflow-x:auto; font-family:Arial, sans-serif;">
      <svg viewBox="0 0 {width} {height}" width="100%" height="auto" role="img" aria-label="Matriz de probabilidades SBM">
        <rect width="100%" height="100%" fill="#ffffff" />
        <text x="24" y="32" font-size="21" font-family="Arial" font-weight="700" fill="#111">Matriz de probabilidades SBM</text>
        <text x="24" y="56" font-size="13" font-family="Arial" fill="#555">Cada celda muestra p_ij = aristas observadas / aristas posibles entre bloques.</text>
        {''.join(labels)}
        {''.join(parts)}
      </svg>
    </div>
    '''
    display(HTML(svg))


## Analisis de la red

La red tiene $|V|=47$ vertices y $|E|=148$ aristas. El numero maximo de aristas posibles en un grafo simple no dirigido con 47 vertices es $\binom{47}{2}=1081$, por lo que la densidad es de $0.137$.

Los vertices de mayor grado son PADME con 30 conexiones, OBI-WAN con 29, ANAKIN con 24, YODA con 14, MACE WINDU con 13, y luego PALPATINE y JAR JAR con 11. Esto sugiere que la red esta organizada alrededor de pocos personajes puente, pero el SBM permite revisar si esas conexiones se concentran en bloques narrativos mas estables.

In [4]:
show_table(graph_summary(G1))
show_table(top_degrees(G1, k=10))


nodos,aristas,densidad,componentes,tamano_componente_mayor,pares_sin_arista
47,148,0.1369,1,47,933


vertice,grado
PADME,30
OBI-WAN,29
ANAKIN,24
YODA,14
MACE WINDU,13
JAR JAR,11
PALPATINE,11
MAS AMEDDA,10
SENATOR ASK AAK,10
BAIL ORGANA,9


## Red original con bloques SBM

La figura colorea los nodos con los bloques inferidos por SBM. El layout usa esos bloques como inicializacion visual: nodos del mismo bloque arrancan cerca entre si y despues se aplica una dinamica de fuerzas para separar regiones de la red.

El color del nodo representa su bloque SBM. El tamano del nodo representa el grado $k_v$, es decir, el numero de coapariciones directas del personaje.

In [5]:
blocks, block_of, sbm_metadata = sbm_blocks(G1)
show_table([{
    "metodo": sbm_metadata["method_label"],
    "graph_tool": sbm_metadata.get("graph_tool_status", "disponible"),
    "K": len(blocks),
    "seleccion_K": sbm_metadata["k_selection"],
    "SBM_bayesiano_exacto": sbm_metadata["is_exact_sbm"],
}])
show_network_svg(G1, title="Red base con bloques SBM")

metodo,graph_tool,K,seleccion_K,SBM_bayesiano_exacto
Fallback espectral + BIC para SBM Bernoulli,graph-tool no disponible o fallo: ModuleNotFoundError: No module named 'graph_tool',5,inferido automaticamente por BIC,False


## Explicacion del modelo

El **Stochastic Block Model** (SBM) supone que cada personaje pertenece a un bloque latente y que la probabilidad de observar una arista depende solamente de los bloques de los dos personajes. Si $z_i$ es el bloque del personaje $i$, entonces

$$P(A_{ij}=1\mid z_i=r,z_j=s)=p_{rs}.$$

La prioridad del notebook es usar `graph_tool.inference.minimize_blockmodel_dl`, que infiere los bloques con un SBM bayesiano mediante minimizacion de description length. En este entorno `graph-tool` no esta disponible, asi que se usa un fallback explicito: una aproximacion de SBM Bernoulli que construye la matriz de adyacencia, obtiene una representacion espectral, agrupa con k-means y selecciona $K$ por BIC sobre la log-verosimilitud del SBM.

Este fallback **no es SBM bayesiano exacto**; es una aproximacion espectral-verosimilitud para inferir bloques compatibles con un SBM Bernoulli. A diferencia de modularidad, los bloques principales no se obtienen maximizando $Q$, sino buscando una particion que explique las probabilidades de arista entre grupos. Despues se calcula

$$\hat p_{rs}=\frac{\text{aristas observadas entre los bloques } r \text{ y } s}{\text{aristas posibles entre los bloques } r \text{ y } s}.$$

In [6]:
sbm_modularity_value = nx.algorithms.community.modularity(G1, blocks)
print(f"Bloques SBM estimados: {len(blocks)} | modularidad de la particion SBM: {sbm_modularity_value:.3f}")
show_table(sbm_metadata.get("k_candidates", []), float_digits=3)
show_table(block_summary_table(G1, blocks), float_digits=3)

Bloques SBM estimados: 5 | modularidad de la particion SBM: 0.170


K,log_likelihood,BIC
2,-397.019,821.980
3,-327.553,710.992
4,-302.167,695.147
5,-251.184,635.095
6,-233.211,648.049
7,-200.789,639.089
8,-226.816,754.014


bloque,nodos,aristas_internas,posibles_internas,p_interna,porcentaje_aristas,personajes
B0,13,9,78,0.115,0.061,"BERU, CLIEGG, DOOKU, JOBAL, OWEN, QUEEN JAMILLIA, RUWEE, RYOO & POOJA, SHMI, SIO BIBBLE, SOLA, THREEPIO, WATTO"
B1,12,8,66,0.121,0.054,"BOBA FETT, CHILDREN, DEXTER JETTSTER, HERMIONE BAGWA, JANGO, JANGO FETT, JOCASTA NU, LAMA SU, MACE, PK-4, TAUN WE, WINDU"
B2,10,33,45,0.733,0.223,"AMIDALA, BAIL ORGANA, JAR JAR, KI-ADI-MUNDI, MACE WINDU, MAS AMEDDA, ORN FREE TAA, PALPATINE, SENATOR ASK AAK, YODA"
B3,8,7,28,0.250,0.047,"CAPTAIN TYPHO, COUNT DOOKU, DORME, ELAN, NUTE GUNRAY, POGGLE, SUN RIT, ZAM WESSEL"
B4,4,3,6,0.500,0.020,"ANAKIN, DARTH SIDIOUS, OBI-WAN, PADME"


## Matriz de probabilidades entre bloques SBM

La matriz SBM muestra que la red no se mezcla de forma homogenea. Si todos los personajes interactuaran como una sola masa, las probabilidades entre bloques serian parecidas a la densidad global de la red, $0.137$. En cambio, las probabilidades internas y cruzadas cambian segun el bloque inferido. Esto permite leer la pelicula como una combinacion de frentes narrativos y personajes puente.

In [7]:
show_table(block_probability_table(G1, blocks), float_digits=3)
show_block_heatmap_svg(G1, blocks)

bloque_i,bloque_j,aristas_observadas,aristas_posibles,probabilidad_sbm
B0,B0,9,78,0.115
B0,B1,1,156,0.006
B0,B2,0,130,0.000
B0,B3,0,104,0.000
B0,B4,22,52,0.423
B1,B0,1,156,0.006
B1,B1,8,66,0.121
B1,B2,5,120,0.042
B1,B3,0,96,0.000
B1,B4,13,48,0.271


## Pares no observados con mayor probabilidad SBM

Para un par sin arista, el SBM no usa vecinos comunes directamente: asigna como puntaje la probabilidad del par de bloques al que pertenece. Por eso hay empates. La lectura importante no es solo el orden exacto, sino el tipo de relacion que el modelo considera plausible: principalmente conexiones dentro del mismo bloque narrativo.

In [8]:
sbm_candidates = sbm_missing_edges(G1, blocks, block_of, k=12)
show_table(sbm_candidates, float_digits=3)
show_network_svg(G1, title="Pares no observados con mayor probabilidad SBM", predicted=sbm_candidates[:8])

u,v,puntaje,bloque_u,bloque_v,tipo
YODA,ORN FREE TAA,0.733,B2,B2,intra-bloque
YODA,AMIDALA,0.733,B2,B2,intra-bloque
SENATOR ASK AAK,KI-ADI-MUNDI,0.733,B2,B2,intra-bloque
ORN FREE TAA,MACE WINDU,0.733,B2,B2,intra-bloque
ORN FREE TAA,KI-ADI-MUNDI,0.733,B2,B2,intra-bloque
ORN FREE TAA,BAIL ORGANA,0.733,B2,B2,intra-bloque
MAS AMEDDA,KI-ADI-MUNDI,0.733,B2,B2,intra-bloque
JAR JAR,KI-ADI-MUNDI,0.733,B2,B2,intra-bloque
JAR JAR,AMIDALA,0.733,B2,B2,intra-bloque
AMIDALA,MACE WINDU,0.733,B2,B2,intra-bloque


## Puentes observados entre bloques

Los enlaces entre bloques son menos probables bajo el SBM, asi que los que si existen son especialmente informativos: conectan subtramas que normalmente aparecen separadas.

In [9]:
show_table(observed_block_bridges(G1, block_of, k=12))

u,v,bloque_u,bloque_v,grado_u+grado_v
PADME,YODA,B4,B2,44
MACE WINDU,PADME,B2,B4,43
OBI-WAN,YODA,B4,B2,43
MACE WINDU,OBI-WAN,B2,B4,42
JAR JAR,PADME,B2,B4,41
PADME,PALPATINE,B4,B2,41
JAR JAR,OBI-WAN,B2,B4,40
MAS AMEDDA,PADME,B2,B4,40
OBI-WAN,PALPATINE,B4,B2,40
PADME,SENATOR ASK AAK,B4,B2,40


## Comparacion con modularidad

La modularidad se conserva como punto de comparacion, no como metodo principal. Aqui se compara la particion inferida por SBM contra `greedy_modularity_communities`: numero de grupos, modularidad $Q$ de cada particion, ARI y NMI. ARI y NMI cercanos a 1 indicarian particiones muy parecidas; valores menores indican que SBM esta separando la red con otro criterio.

In [10]:
modularity_blocks, modularity_of = modularity_partition(G1)
show_table(comparison_table(G1, blocks, block_of, modularity_blocks, modularity_of, sbm_metadata), float_digits=3)
show_table([{
    "comunidad_modularidad": f"M{i}",
    "nodos": len(block),
    "personajes": ", ".join(block),
} for i, block in enumerate(modularity_blocks)])

particion,metodo,bloques,modularidad_Q,ARI_vs_modularidad,NMI_vs_modularidad
SBM,Fallback espectral + BIC para SBM Bernoulli,5,0.170,0.512,0.584
Modularidad,greedy_modularity_communities,4,0.387,1.000,1.000


comunidad_modularidad,nodos,personajes
M0,16,"ANAKIN, BERU, CAPTAIN TYPHO, CLIEGG, DORME, JOBAL, OWEN, PADME, QUEEN JAMILLIA, RUWEE, RYOO & POOJA, SHMI, SIO BIBBLE, SOLA, THREEPIO, WATTO"
M1,13,"AMIDALA, BAIL ORGANA, CHILDREN, JAR JAR, KI-ADI-MUNDI, MACE, MACE WINDU, MAS AMEDDA, ORN FREE TAA, PALPATINE, SENATOR ASK AAK, WINDU, YODA"
M2,13,"BOBA FETT, DEXTER JETTSTER, DOOKU, ELAN, HERMIONE BAGWA, JANGO, JANGO FETT, JOCASTA NU, LAMA SU, OBI-WAN, PK-4, TAUN WE, ZAM WESSEL"
M3,5,"COUNT DOOKU, DARTH SIDIOUS, NUTE GUNRAY, POGGLE, SUN RIT"


## Analisis de resultados

El notebook intento primero usar `graph_tool.inference.minimize_blockmodel_dl`, pero en este entorno no esta instalado `graph-tool`. Por eso se uso el fallback explicito: clustering espectral sobre la matriz de adyacencia, ajuste de un SBM Bernoulli y seleccion automatica de $K$ por BIC. El metodo no es SBM bayesiano exacto; es una aproximacion espectral-verosimilitud. Con ese criterio, el mejor valor fue $K=5$ porque tuvo el BIC mas bajo: $635.095$.

El SBM aproximado encontro 5 bloques. **B0** contiene BERU, CLIEGG, DOOKU, JOBAL, OWEN, QUEEN JAMILLIA, RUWEE, RYOO & POOJA, SHMI, SIO BIBBLE, SOLA, THREEPIO y WATTO. Es un bloque periferico de entorno familiar/domestico y Naboo-Tatooine. Su probabilidad interna es baja ($0.115$), pero se conecta con fuerza con B4 ($p_{0,4}=0.423$). En terminos de trama, esto sugiere que estos personajes no forman un frente autosuficiente; funcionan como contexto alrededor de los protagonistas, sobre todo ANAKIN y PADME.

**B1** contiene BOBA FETT, CHILDREN, DEXTER JETTSTER, HERMIONE BAGWA, JANGO, JANGO FETT, JOCASTA NU, LAMA SU, MACE, PK-4, TAUN WE y WINDU. Representa el frente de investigacion y pistas: Kamino, Jango, Boba, Dexter y archivos Jedi. Su densidad interna tambien es baja ($0.121$), pero tiene una relacion clara con B4 ($p_{1,4}=0.271$), porque OBI-WAN es quien atraviesa ese circuito narrativo.

**B2** es el bloque politico/Jedi compacto: AMIDALA, BAIL ORGANA, JAR JAR, KI-ADI-MUNDI, MACE WINDU, MAS AMEDDA, ORN FREE TAA, PALPATINE, SENATOR ASK AAK y YODA. Tiene la mayor probabilidad interna de la particion ($0.733$). Este es el frente institucional de la pelicula: Senado, Consejo Jedi y decision politica sobre la Republica. Por eso los pares no observados con mayor probabilidad SBM aparecen dentro de B2, como ORN FREE TAA--YODA, AMIDALA--YODA o AMIDALA--MACE WINDU.

**B3** contiene CAPTAIN TYPHO, COUNT DOOKU, DORME, ELAN, NUTE GUNRAY, POGGLE, SUN RIT y ZAM WESSEL. Mezcla escoltas/amenaza inicial con el nucleo separatista visible. Su densidad interna es $0.250$, pero lo mas importante es su conexion con B4: $p_{3,4}=0.719$, la probabilidad cruzada mas alta de la matriz. Narrativamente, esto refleja que la conspiracion y los antagonistas se definen por su contacto con los protagonistas, no por estar aislados en un solo bloque cerrado.

**B4** contiene solo ANAKIN, DARTH SIDIOUS, OBI-WAN y PADME. Es pequeno, pero estructuralmente central: tiene probabilidad interna $0.500$ y conexiones muy altas con B2 ($0.500$), B3 ($0.719$) y B0 ($0.423$). El resultado no debe leerse como que estos cuatro forman una subtrama unica, sino como que el SBM los detecta como conectores de bloques. ANAKIN, OBI-WAN y PADME cruzan romance, politica, investigacion y conflicto separatista; DARTH SIDIOUS aparece como una senal estructural de la conspiracion que atraviesa la pelicula.

La comparacion con modularidad confirma que ambos metodos no estan diciendo exactamente lo mismo. Modularidad encuentra 4 comunidades y obtiene $Q=0.387$; la particion SBM aproximada encuentra 5 bloques y su modularidad es menor, $Q=0.170$. La similitud es intermedia: ARI $0.512$ y NMI $0.584$. Esto significa que SBM coincide parcialmente con modularidad, sobre todo en el bloque politico/Jedi, pero separa de otra forma a los protagonistas y a los personajes perifericos.

Para interpretar la trama, modularidad refleja mejor comunidades narrativas compactas tradicionales: romance/familia, Senado/Jedi, investigacion de Obi-Wan y separatistas. El SBM aproximado refleja mejor el patron probabilistico de coaparicion: identifica un bloque institucional muy denso y separa un bloque central de personajes puente que conectan casi todos los frentes. Por eso el SBM es especialmente util para ver la arquitectura de circulacion de la pelicula: no solo que subtramas existen, sino que ANAKIN, PADME y OBI-WAN funcionan como corredores entre esas subtramas.

**Reporte final.** Metodo usado: fallback espectral + BIC para SBM Bernoulli, porque `graph-tool` no esta disponible. Bloques encontrados: 5. $K$ no fue fijado manualmente; fue inferido automaticamente por BIC entre $K=2$ y $K=8$. Diferencia contra modularidad: SBM encontro 5 bloques con $Q=0.170$, mientras modularidad encontro 4 comunidades con $Q=0.387$; ARI $0.512$ y NMI $0.584$.